# Bài học 21: Mở rộng sản phẩm

## Thêm Agent Hiệu đính

Trong bài học cuối cùng này, chúng ta sẽ đi qua một kịch bản thực tế: **thêm agent thứ 5 vào pipeline** — agent hiệu đính kiểm tra ngữ pháp, độ dễ đọc, và SEO best practice trước khi bài viết được lưu.

Chúng ta sẽ không chạy Claude Code thực sự. Thay vào đó, chúng ta sẽ lần theo từng bước của quy trình phát triển từ Bài học 13, và bạn sẽ **kiểm chứng kết quả** bằng kiến thức đã học.

Đây là cách bạn sẽ làm việc trong thực tế:
1. Bạn mô tả điều mình muốn
2. Claude Code triển khai
3. Bạn kiểm chứng kết quả bằng kiến thức của mình

## Bước 1: Tìm hiểu

Trước khi thực hiện bất kỳ thay đổi nào, Claude Code sẽ khám phá codebase. Đây là prompt bạn sẽ đưa ra:

> "Tôi muốn thêm agent hiệu đính vào pipeline. Trước khi thay đổi gì, hãy khám phá codebase và giải thích: (1) các bước pipeline được định nghĩa ở đâu, (2) agent mới nên thêm vào đâu, (3) cần tuân theo pattern schema nào."

Claude Code sẽ đọc:
- `output/pipeline.py` — để hiểu trình tự các bước
- `output/agents/` — để xem cách các agent được định nghĩa và pattern schema

Hãy tự làm tương tự — đọc các file quan trọng:

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output"))

# Đọc pipeline để hiểu bước mới sẽ nằm ở đâu
pipeline_path = os.path.abspath("../../output/pipeline.py")
with open(pipeline_path, "r", encoding="utf-8") as f:
    pipeline_code = f.read()

print(f"pipeline.py ({len(pipeline_code.splitlines())} dòng)\n")
print(pipeline_code)

In [ ]:
agents_path = os.path.abspath("../../output/agents/schemas.py")
with open(agents_path, "r", encoding="utf-8") as f:
    agents_code = f.read()

print(f"agents/schemas.py ({len(agents_code.splitlines())} dòng)\n")
print(agents_code)

## Bước 2: Lên kế hoạch

Sau khi hiểu codebase, bạn sẽ nói với Claude Code:

> "Lên kế hoạch thêm agent hiệu đính giữa agent viết bài và agent hình ảnh. Agent hiệu đính cần kiểm tra ngữ pháp, độ dễ đọc, cách sử dụng từ khóa, và SEO best practice. Trả về kết quả có cấu trúc gồm bài viết đã sửa và danh sách vấn đề tìm được. Dùng Claude Sonnet vì cần structured output."

Kế hoạch tốt sẽ bao gồm:
1. **Schema mới** trong `agents/schemas.py`: `ProofreadIssue` + `ProofreadResult`
2. **Agent mới** trong `agents/proofreader.py`: `proofreader_agent` dùng Claude Sonnet + output_schema
3. **Bước pipeline mới** trong `pipeline.py`: giữa "writing" và "enriching"
4. **Trạng thái mới**: "proofreading" giữa "writing" và "enriching"

Hãy xây dựng và kiểm chứng từng phần:

## Bước 3: Triển khai — Schema

Việc đầu tiên Claude Code tạo là schema. Tuân theo pattern từ `agents/schemas.py` (và những gì bạn đã học trong Bài học 10 và 12), schema hiệu đính sẽ trông như thế này:

In [ ]:
from pydantic import BaseModel, Field

# Vấn đề tìm được khi hiệu đính (model lồng nhau — cùng pattern với OutlineSection)
class ProofreadIssue(BaseModel):
    issue_type: str = Field(description="Type: grammar, readability, seo, factual")
    location: str = Field(description="Which section or paragraph has the issue")
    description: str = Field(description="What the issue is")
    suggestion: str = Field(description="How to fix it")
    severity: str = Field(default="medium", description="low, medium, or high")

# Kết quả hiệu đính đầy đủ (cùng pattern với ContentOutline)
class ProofreadResult(BaseModel):
    corrected_article: str = Field(description="The full article with corrections applied")
    issues_found: list[ProofreadIssue] = Field(description="List of issues found and fixed")
    readability_score: int = Field(description="Readability score 1-10 (10 = easiest to read)")
    seo_score: int = Field(description="SEO optimization score 1-10")
    word_count: int = Field(description="Word count of corrected article")

# Kiểm tra schema hoạt động (không gọi API — chỉ validation bằng Pydantic)
test_result = ProofreadResult(
    corrected_article="# Test Article\n\nThis is a corrected article.",
    issues_found=[
        ProofreadIssue(
            issue_type="grammar",
            location="Paragraph 2",
            description="Run-on sentence",
            suggestion="Split into two sentences",
            severity="medium",
        ),
        ProofreadIssue(
            issue_type="seo",
            location="Introduction",
            description="Target keyword missing from first paragraph",
            suggestion="Add 'on-page SEO' to the opening sentence",
            severity="high",
        ),
    ],
    readability_score=7,
    seo_score=8,
    word_count=150,
)

print("Validation schema thành công!\n")
print(f"Bài viết đã sửa: {test_result.corrected_article[:50]}...")
print(f"Số vấn đề tìm được: {len(test_result.issues_found)}")
for issue in test_result.issues_found:
    print(f"  [{issue.severity.upper()}] {issue.issue_type}: {issue.description}")
print(f"Khả năng đọc: {test_result.readability_score}/10")
print(f"Điểm SEO: {test_result.seo_score}/10")
print(f"Số từ: {test_result.word_count}")

## Bước 3: Triển khai — Agent

Tiếp theo, Claude Code sẽ thêm agent vào file mới `agents/proofreader.py`. Tuân theo pattern hiện có:

In [ ]:
# Đây là code mà Claude Code sẽ thêm vào agents/proofreader.py
# (Chúng ta không chạy thật — chỉ xem xét code)

proofreader_code = '''
# Proofreader Agent -- kiểm tra ngữ pháp, độ dễ đọc, SEO best practice
#    Model: Claude Sonnet | Tools: không | Output: ProofreadResult schema
proofreader_agent = Agent(
    name="Proofreader Agent",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    output_schema=ProofreadResult,
    instructions=[
        "You are an expert content editor and SEO proofreader.",
        "Given a Markdown article and its target keywords, proofread the article thoroughly.",
        "Check for: grammar errors, awkward phrasing, readability issues, "
        "keyword usage and density, SEO best practices (title, headings, meta description).",
        "Fix all issues directly in the article text.",
        "Return the corrected article along with a list of every issue you found and fixed.",
        "Rate readability (1-10) and SEO optimization (1-10).",
    ],
)
'''

print("Định nghĩa agent mà Claude Code sẽ tạo:")
print("=" * 60)
print(proofreader_code)

## Bước 3: Triển khai — Bước Pipeline

Cuối cùng, Claude Code sẽ thêm bước mới vào `pipeline.py` giữa writing và enriching:

```python
# Trong pipeline.py, sau bước writing:

# --- Bước 4: Hiệu đính ---
update_article_status(article_id, "proofreading")
proofread_response = proofreader_agent.run(
    f"Proofread this article. Target keywords: {keywords}\n\n{article_markdown}"
)
proofread_result = proofread_response.content
article_markdown = proofread_result.corrected_article
# Lưu dữ liệu hiệu đính vào database...
```

Luồng pipeline trở thành:
```
queued → researching → outlining → writing → proofreading → enriching → review
```

## Bước 4: Kiểm chứng

Đây là lúc kiến thức CỦA BẠN phát huy tác dụng. Dùng mọi thứ đã học để kiểm chứng phần triển khai:

In [ ]:
# Danh sách kiểm chứng — áp dụng kiến thức từ tất cả các mô-đun

checks = [
    ("Mô-đun 2 (Bài 5)", "Agent có vấn đề về knowledge cutoff không?",
     "Không — hiệu đính chỉ cần nội dung bài viết, không cần thông tin real-time. Không cần tools."),
    
    ("Mô-đun 2 (Bài 6)", "Instructions có được cấu trúc tốt không?",
     "Có — Vai trò ('expert editor'), Nhiệm vụ ('proofread'), Ràng buộc ('check grammar, SEO...')."),
    
    ("Mô-đun 2 (Bài 7)", "Lựa chọn model có đúng không?",
     "Đúng — Claude Sonnet. Cần output_schema (ProofreadResult). Không cần tools. Grok không hỗ trợ schema, nên Claude là lựa chọn đúng."),
    
    ("Mô-đun 3 (Bài 10)", "Pattern schema có đúng không?",
     "Đúng — ProofreadIssue được lồng trong ProofreadResult, giống OutlineSection trong ContentOutline."),
    
    ("Mô-đun 3 (Bài 12)", "Truy cập dữ liệu lồng nhau có hoạt động không?",
     "Có — result.issues_found[0].description hoạt động. Cùng pattern với outline.sections[0].heading."),
    
    ("Mô-đun 4 (Bài 16)", "Bước pipeline có tuân theo pattern không?",
     "Có — cập nhật trạng thái, chạy agent, lưu kết quả. Giống các bước khác."),
    
    ("Mô-đun 5 (Bài 18)", "Cập nhật trạng thái Airtable có đúng không?",
     "Cần thêm 'proofreading' làm trạng thái hợp lệ. Kiểm tra luồng trạng thái đã được cập nhật."),
    
    ("Mô-đun 5 (Bài 19)", "Kiến trúc có vẫn hợp lý không?",
     "Có — agent mới nằm gọn trong chuỗi bước pipeline.py. Không cần thay đổi chat.py hay tools/workspace.py."),
]

print("=== Danh sách kiểm chứng ===\n")
all_pass = True
for module, question, answer in checks:
    print(f"  [{module}]")
    print(f"  H: {question}")
    print(f"  Đ: {answer}")
    print()

## Thêm ý tưởng mở rộng

Đây là những thứ khác bạn có thể yêu cầu Claude Code xây dựng, xếp theo độ khó:

| Mở rộng | Độ khó | Prompt cho Claude Code |
|---------|--------|------------------------|
| Thay đổi giọng văn | Dễ | "Cập nhật instructions của writer_agent trong agents/writer.py để viết theo phong cách thân thiện, gần gũi" |
| Thêm kiểm tra mật độ từ khóa | Trung bình | "Thêm trường keyword_density vào ProofreadResult trong agents/schemas.py để tính tần suất xuất hiện của từ khóa mục tiêu" |
| Thêm agent dịch thuật | Trung bình | "Thêm agent thứ 6 vào pipeline trong agents/translator.py để dịch bài viết hoàn chỉnh sang tiếng Việt bằng Claude Sonnet" |
| So sánh AIO theo thời gian | Trung bình | "Thêm tool so sánh kết quả AIO giữa hai lần kiểm tra để phát hiện thay đổi trong AI Overview" |
| Thêm tool Google Search Console | Khó | "Tạo toolkit GSC mới trong output/tools/ theo pattern của FreepikTools trong agents/image.py, rồi thêm vào research agent" |
| Thêm dashboard web | Khó | "Tạo dashboard web đơn giản bằng Flask hiển thị tất cả bài viết và kết quả phân tích AIO từ Airtable" |

Mỗi mở rộng đều tuân theo cùng quy trình: Tìm hiểu -> Lên kế hoạch -> Triển khai -> Kiểm chứng.

> **Đã có sẵn:** Sản phẩm hỗ trợ **xử lý hàng loạt song song** ngay từ đầu. Trong chat, chỉ cần nói "Tạo bài viết về Topic 1, Topic 2, Topic 3 song song". Tính năng này dùng `ThreadPoolExecutor` của Python — giống như có nhiều công nhân trong nhà máy thay vì chỉ một người.

## Chu trình phát triển

```
  ┌─────────┐
  │ Ý tưởng │  "Tôi muốn thêm hiệu đính"
  └────┬────┘
       ▼
  ┌──────────┐
  │ Kế hoạch │  Claude Code khám phá, bạn duyệt phương án
  └────┬─────┘
       ▼
  ┌──────────┐
  │ Viết code│  Claude Code viết code
  └────┬─────┘
       ▼
  ┌────────────┐
  │ Kiểm chứng │  BẠN kiểm tra bằng kiến thức của mình
  └────┬───────┘
       ▼
  ┌─────────┐
  │Đưa vào  │  Chạy thử, sử dụng, lặp lại
  │ dùng    │
  └────┬────┘
       │
       └──────→ (quay lại Ý tưởng cho lần cải tiến tiếp theo)
```

**Bạn là người kiểm soát chất lượng. Claude Code là người xây dựng. Kiến thức của bạn là thứ giúp bạn làm việc hiệu quả.**

Nếu không có kiến thức từ khóa học này, bạn không thể kiểm chứng lựa chọn model có đúng không, schema có tuân theo pattern không, hay bước pipeline có được đặt đúng chỗ không. Giờ thì bạn có thể.

## Chúc mừng! Bạn đã hoàn thành tất cả 21 bài học!

Đây là toàn bộ hành trình:

### Mô-đun 1: Nền tảng Python (Bài học 1-4)
Biến, list, dictionary, hàm, package — nền tảng để đọc hiểu code.

### Mô-đun 2: Hiểu về AI (Bài học 5-7)
Cách LLM hoạt động, prompt và ngữ cảnh, lựa chọn model — lý do đằng sau mọi quyết định.

### Mô-đun 3: Xây dựng Agent (Bài học 8-12)
Agent đầu tiên, tools, structured output, chuỗi agent, mini pipeline — thực hành xây dựng agent.

### Mô-đun 4: Xây dựng sản phẩm (Bài học 13-17)
Claude Code, agent nghiên cứu, dàn ý, viết bài, hình ảnh — từ công cụ đến pipeline.

### Mô-đun 5: Sản phẩm hoàn chỉnh (Bài học 18-21)
Database (Airtable), cách mọi thứ kết nối, giao diện chat, mở rộng sản phẩm.

---

### Bảng ánh xạ Bài học - File Production (Cập nhật)

| Bài học | Khái niệm | File production |
|---------|-----------|------------------|
| 05-07 | LLM, prompt, model | (Lý thuyết — định hướng mọi quyết định) |
| 08-09 | Tạo agent, tools | `output/agents/researcher.py`, `outliner.py`, `writer.py`, `image.py` |
| 10, 14 | Pydantic schema | `output/agents/schemas.py` |
| 11-12, 14-17 | Chuỗi agent, pipeline | `output/pipeline.py` |
| 09, 15 | Custom toolkit | `output/agents/image.py` |
| 13 | Claude Code | `CLAUDE.md` (bản thiết kế cho Claude Code) |
| 18 | Database (Airtable) | `output/tools/airtable.py` |
| 19 | Cách mọi thứ kết nối | Tất cả file — toàn bộ luồng xử lý |
| 20 | Giao diện chat, workspace tools | `output/chat.py`, `output/tools/workspace.py` |
| 21 | Mở rộng sản phẩm | (Bài học này — kỹ năng kiểm chứng) |

---

### Tiếp theo là gì?

1. **Bắt đầu sử dụng sản phẩm** — chạy `python output/chat.py` và tạo bài viết cho team
2. **Tùy chỉnh instructions của agent** — điều chỉnh phong cách viết, giọng văn, chiến lược SEO
3. **Thử Claude Code** — cài đặt và thực hiện thay đổi nhỏ (ví dụ: đổi giọng văn của writer)
4. **Xây dựng tools riêng** — thêm tools mà team SEO cần (GSC, Analytics, v.v.)
5. **Dạy người khác** — giờ bạn đã hiểu đủ để hướng dẫn đồng nghiệp

**Cách tốt nhất để học là xây dựng. Bắt đầu nhỏ, kiểm chứng cẩn thận, lặp lại nhanh.**

Chúc bạn thành công!

In [ ]:
# Bài tập cuối: Khám phá toàn bộ cấu trúc dự án

import os

output_dir = os.path.abspath("../../output")
print("Codebase production của bạn:\n")

for root, dirs, files in os.walk(output_dir):
    # Bỏ qua __pycache__
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    level = root.replace(output_dir, "").count(os.sep)
    indent = "  " * level
    folder_name = os.path.basename(root)
    print(f"{indent}{folder_name}/")
    sub_indent = "  " * (level + 1)
    for file in sorted(files):
        if file.endswith(".py"):
            filepath = os.path.join(root, file)
            with open(filepath, "r", encoding="utf-8") as f:
                lines = len(f.readlines())
            print(f"{sub_indent}{file} ({lines} dòng)")

print("\nBạn hiểu rõ từng file trong số này.")
print("Kiến thức đó là siêu năng lực của bạn khi làm việc với Claude Code.")